In [ ]:
## IMPORTANT: On Colab, we expect your lab to be in the ee345 folder
## Please contact staff if you encounter any problems with installing dependencies
import sys
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/ee345/labs/w7
    %pip install -r ./requirements.txt

# Install required packages with compatible versions
!pip install -q datasets "plotly>=6.1.1" "kaleido>=0.2.1"

import plotly.io as pio
pio.renderers.default = pio.renderers.default + "+png"

# Week 7 M3-N1: Welcome to the Arena (Warmup)

please submit the printed notebook (.pdf) and the notebook (.ipynb) itself on gradescope

Please put down your name(s) here: (you must enter your team members' names on gradescope too)

[ Your name ]

---

In this notebook you will get more experience with logistic regression in two very different settings: creating leaderboards and predicting model responses.

We will be taking real data from [LMArena](https://lmarena.ai/), a popular platform for crowsourcing evaluations of large language models and recreating their leaderboards, with a few fun extra steps along the way.

The chats can be viewed interactively by accessing [ChatBot-Arena-Viewer](https://huggingface.co/spaces/BerkeleyML/Chatbot-Arena-Viewer) through hugging face. Much of the first half of this homework was first written by Prof Gonzalez back when his students first started the project, and now LMArena is a standard evaluation for large language models and turned into a company! Don't let anyone tell you logistic regression isn't valuable, it's worth at least $600 Million.

---

## Notebook Roadmap

This notebook is organized into the following sections:

### Part 1: Data Exploration and Understanding (Q1)
- **Q1a**: Identify top 20 models by battle count
- **Q1b**: Filter battles to selected models and remove ties
- Explore battle distributions and understand the LMArena dataset

### Part 2: Model Rankings via Win Rates (Q2)
- **Q2a**: Compute pairwise win fractions between models
- **Q2b**: Visualize win fraction heatmaps
- **Q2c**: Analyze average win rates, parameter sizes, and limitations of simple win-rate metrics

### Part 3: Prompt Analysis and Leaderboard Shifts (Q3)
- **Q3a**: Compare leaderboards with and without top 10 frequent prompts
- **Q3b**: Analyze category-specific performance (code, math, creativity, etc.)
- **Q3c**: Use K-Means clustering to discover prompt topics via TF-IDF
- **Q3d**: Interpret clustering results and identify prompt patterns
- **Q3e**: Find custom subset of prompts that causes top 5 ranking changes

---

### Key Learning Objectives

1. Learn how to evaluate large language models (LLMs) using pairwise comparison data from LMArena
2. Analyze battle distributions and compute win rates
3. Find subsets of prompts which result in changes to the leaderboard

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
pd.options.plotting.backend = "plotly"
#set fixed seed of 189
np.random.seed(189)

## What is LMArena?

![LMArena](https://i.imgur.com/tbrkWVX.png)

LMArena (previously known as Chatbot Arena) is a platform that evaluates generative models (e.g. chatbots or image generation models) through anonymous, crowd-sourced pairwise comparisons. Users enter prompts for two anonymous models to respond to and vote on the model that gave the better response, in which the model's identities are revealed (shown below). Users can also choose models to test themselves, but for the purposes of this lab we will only focus on the anonymous side-by-side comparisons, which we call **"battles"** - since those are what are used to calculate the leaderboard.

![LMArena Example](https://i.imgur.com/rgv0jCb.png)

In this lab we will investigate what these battles look like, how we can use these pairwise comparisons to get a leaderboard, and how we can find certain features of model responses that have an influence on preference.

Although it is not required for this notebook, the [Chatbot Arena paper](https://arxiv.org/abs/2403.04132) can provide good intuition on how to answer the free response questions.

### Download Data

First, let's load a set of publicly released arena battles from [Hugging Face](https://huggingface.co/datasets/lmarena-ai/arena-human-preference-100k) — a popular website for sharing machine learning datasets and models.

**Note**: Before you get started with the lab, we recommend you make a [huggingface account](https://huggingface.co/welcome) to play around with data visualization apps! It is also a great general hub for downloading the majority of popular datasets in machine learning. Once you make the account, generate the token for login [huggingface token](https://huggingface.co/docs/hub/security-tokens). This should be similar to how you generate a token for your git account.

In [ ]:
# To log into huggingface, uncomment the line below and re-run the cell

#! pip install ipywidgets
#from huggingface_hub import notebook_login
#notebook_login()

In [ ]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset (this will take a few minutes to download)
# if you don't have an account it may throw a warning to create a HF_TOKEN but you should still have access to the dataset if you skip this
ds = load_dataset("lmarena-ai/arena-human-preference-100k")
battles = ds['train'].to_pandas()

In [ ]:
Now let's look at the format of this data.

Printing out the first row we see there are many fields with the most important being:

- **question_id (str)** - the ID of that battle
- **model_a, model_b (str)** - the models which participated in this battle
- **winner (str)** - which response the user preferred: can be `model_a`, `model_b`, `tie`, and `tie (bothbad)`
- **conversation_a (dict)** - the conversation between the user and model a
- **conversation_b (dict)** - the conversation between the user and model b (note that the user turns in conversation_a and conversation_b are the same since this is a side by side comparison)
- **turn (int)** - number of turns in the conversation (1 turn means the user asked 1 question, 2 turns means the user asked a question, got an answer, then asked another question, got the response, then voted)
- **language (str)** - the language of the user prompt

We also have some other columns that may be useful for us later:

- **is_code (bool)** - either the prompt, the response, or both contains code
- **is_refusal (bool)** - one of the models refused to answer (usually this is because the model thinks it would be unethical to answer)
- **dedup_tag (dict)** - indicates whether the prompt appears very often (high_frequency) and if it does, whether it will be sampled (subsampled). We subsample these high frequency prompts so that common questions don't overly influence the leaderboard.
- **category_tag (dict)** - tags for question type (e.g. math and instruction following). These are assigned via an LLM labeler, more details on what the categories are in this [blog post](https://blog.lmarena.ai/blog/2024/arena-category/) (the criteria tags correspond to the hard prompts category described in the blog post).

In [ ]:
battles.head(5)

In [ ]:
example = battles.iloc[4]
print(f"Conversation A (model = {example['model_a']}):")
print(example['conversation_a'])
print(f"Conversation B (model = {example['model_b']}):")
print(example['conversation_b'])
print("Category Tag:")
print(example['category_tag'])

### Exploratory Analysis

Before we get into leaderboard calculation, let's first conduct some basic exploratory analysis to highlight a few key properties and caveates with this data.

In [ ]:
battles.winner.hist(title="Counts of Battle Outcomes", text_auto=True)

#### NOTE: Notice how LMArena has two types of ties: tie (both bad) and just tie.

### Battle Counts

We see that certain models participate in more battles. This is due to two reasons:
1. Several different matching and sampling algorithms were used. LMArena employs weighted sampling methods, which assign greater weights to better models.
2. Since models are added to the arena when they come out, some models have been on the arena for many months while others have only been on for a few weeks.

In [ ]:
fig = pd.concat([battles["model_a"], battles["model_b"]]).value_counts().plot.bar(title="Battle Count for Each Model", text_auto=True)
fig.update_layout(xaxis_title="Model", yaxis_title="Battle Count", height=400, showlegend=False)
fig

# Question 1: Data Exploration

## Question 1a
Since it can be hard to reason over that many models, we want to look at the top 20 models by battle count.

**Task:**
Return the a list of top 20 models by battle count for both `model_a` and `model_b` combined, SORTED by the battle count.

In [ ]:
# TODO:
selected_models = ...

**Hint:** You may find it helpful to use a boolean array.

In [ ]:
from typing import Tuple, Set
import pandas as pd

def subselect_battles(
    battles: pd.DataFrame,
    selected_models: Set[str]
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    # Filters the battles DataFrame to only include battles between the selected models.
    # Returns a tuple of the dataframe filtered by models and the dataframe filtered my models with ties removed.
    # TODO: Implement this section
    return selected_battles, selected_battles_no_ties

selected_battles, selected_battles_no_ties = subselect_battles(battles, selected_models)

## Question 1b

**Task:**
Now, let's filter out and select the battles that are between the top 20 models we got from question 1a. Fill in the `subselect_battles` function to return the battles dataframe containing only the selected models and the battles dataframe containing the selected models with ties removed.

In [ ]:
def visualize_battle_count(battles, title, show_num_models=30):
    """
    Input:
        battles : pd.DataFrame with columns ['model_a','model_b', ...]
        title   : str, title for the plot
        show_num_models : int, how many top models (by total battle count) to display

    Output:
        fig : plotly.graph_objects.Figure heatmap of symmetric battle counts
    """
    ptbl = pd.pivot_table(battles, index="model_a", columns="model_b", aggfunc="size", fill_value=0)
    battle_counts = ptbl + ptbl.T
    ordering = battle_counts.sum().sort_values(ascending=False).index
    ordering = ordering[:show_num_models]

    fig = px.imshow(
        battle_counts.loc[ordering, ordering],
        title=title,
        text_auto=True
    )
    fig.update_layout(
        xaxis_title="Model B",
        yaxis_title="Model A",
        xaxis_side="top",
        height=1000,
        width=1000,
        title_y=0.07,
        title_x=0.5,
        font=dict(size=10)
    )
    fig.update_traces(
        hovertemplate="Model A: %{y}<br>Model B: %{x}<br>Count: %{z}<extra></extra>"
    )
    return fig

### Visualize All Selected Battles

In [ ]:
visualize_battle_count(selected_battles, title="Battle Count of Each Combination of Models", show_num_models=30)

### Visualize Selected Battles No Ties

In [ ]:
visualize_battle_count(selected_battles_no_ties, "Battle Count for Each Combination of Models (without Ties)")

### Understanding Battle Distribution

We see many battles between top models (e.g., Claude, GPT, Gemini), while smaller models (e.g., Llama-3-8B) have fewer battles. This is because LMArena employs weighted sampling methods, which assign greater weights to better models.

**Why pair strong models vs. strong models more often?**

LMArena pairs strong models against each other more frequently because the greatest ranking uncertainty exists among top-tier systems. When comparing a strong model to a weak model, the outcome is predictable with high confidence after relatively few battles. However, differentiating between two strong models requires significantly more data to achieve statistical significance, as the performance differences are smaller and more subtle. By focusing sampling resources on strong vs. strong matchups, LMArena efficiently allocates battles where they provide the most information for refining the leaderboard rankings, particularly at the top where users care most about precise distinctions between competitive models.

In [ ]:
lang_counts_all = battles["language"].value_counts()

fig_lang_all = px.bar(
    lang_counts_all,
    title="Distribution of Languages",
    text_auto=True,
    height=400
)
fig_lang_all.update_layout(
    xaxis_title="Language",
    yaxis_title="Count",
    showlegend=False
)
fig_lang_all.show()

## Number of Conversation Turns

Now let's also try to explore conversation turns.

In [ ]:
fig = px.histogram(battles["turn"],
             title=f"Number of Conversation Turns",
             text_auto=True, height=400, log_y = True)
fig.update_layout(xaxis_title="Turns", yaxis_title="Count", showlegend=False)
fig.update_traces(marker_line_color='black', marker_line_width=1)
fig

Now that we have explored our data, let's consider how to use these pairwise battles to rank the models by preference. Our goal is to assign a "strength" parameter to each model that quantifies how likely it is to win against others.

Given we analyzed $M=20$ models and $N=40k$ battles ($26k$ excluding ties), we want to estimate a skill parameter $S_m$ for each model $m \in \{1, \ldots, M\}$. This parameter $S_m$ should reflect the overall ability of model $m$ to be preferred over other models.

Before we move on to more sophisticated probabilistic models that estimate these strength parameters, let’s build some intuition by starting with a simpler metric: the average win rate.

Average win rate is simple: a model's average win rate is the proportion of battles they competed in which resulted in them winning.

## Question 2a

LMArena defines the win rate for a model as the average fraction of times it defeats another model across all its match-ups.

**Task:**
Implement the `compute_pairwise_win_fraction` function, which:

1. Calculates the fraction of times each model beats each other model across all battles.

2. Returns a square DataFrame where entry (i, j) is the fraction of times model i beats model j. The colmns should be the select model names as well as the index (similar to a confusion matrix). Any model pairings which do not have any battles should be given a NaN value. For instance, diagonal of `row_beats_col` should be NaN as no battles exist between a model and itself.

3. The rows and columns of your `row_beats_col` dataframe should be ordered by their average win rate against all other models (i.e. order from strongest to weakest models)

Tips:
* Do not use your `selected_models` variable in your function, otherwise you may run into autograder issues. Instead define a variable which is the list of models in you input `battles` dataframe.

In [ ]:
def compute_pairwise_win_fraction(battles):
    # TODO:

    return row_beats_col


## Question 2b


Let’s visualize how often **Model A** beats **Model B** in non-tied battles. Below we have used your `compute_pairwise_win_fraction` to create heatmap where each cell `(A, B)` displays the **fraction of A’s wins** over B. No TODOs or code to fill in here, this question allows us to viusally inspect your function in the Coding PDF assignment on gradescope.

In [ ]:
def visualize_pairwise_win_fraction(battles, title):
    """
    Input:
        battles : pd.DataFrame of non-tied battles with ['model_a','model_b','winner', ...]
        title   : str
    Output:
        fig : plotly Figure heatmap (cell (A,B) = fraction A beats B)
    """
    row_beats_col = compute_pairwise_win_fraction(battles)
    fig = px.imshow(
        row_beats_col,
        color_continuous_scale='RdBu',
        text_auto=".2f",
        title=title
    )
    fig.update_layout(
        xaxis_title=" Model B: Loser",
        yaxis_title="Model A: Winner",
        xaxis_side="top",
        height=900,
        width=900,
        title_y=0.07,
        title_x=0.5
    )
    fig.update_traces(
        hovertemplate="Model A: %{y}<br>Model B: %{x}<br>Fraction of A Wins: %{z}<extra></extra>"
    )
    return fig


fig = visualize_pairwise_win_fraction(
    selected_battles_no_ties,
    title="Fraction of Model A Wins for All Non-tied A vs. B Battles"
)
fig


Now that we’ve computed and visualized the average win rate of each model against all others, we can start reasoning about the results.
In the chart above, we see that some models have very similar average win rates. For example, some GPT, Claude, and Llama variants sit close together. On the other hand, smaller models like llama-3-8b and gemma-2-9b fall noticeably behind.

**Task: Answer the following questions in the cell below the question cell**


# Question 3: Prompt Analysis

We have explored the general features of this dataset. When evaluating models, it’s also often useful to understand the types of questions that are asked. By grouping similar prompts together, we can analyze which models perform well on certain categories and poorly on others. This helps uncover biases in leaderboards (e.g., a model may excel at coding questions but struggle with creative writing).

**First, let's identify the most frequent prompts.**

In the code cell below, we've already done the following:
*   Extracted the first user message (prompt) from conversation_a
*   Filtered out only the battles in English and only kept the select models using your `subselect_battles` function from 1b

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD

def first_user_text(conv):
    return (conv[0].get("content") or "").strip()

battles['prompt'] = battles['conversation_a'].apply(first_user_text).fillna("")
eng_battles = battles[(battles['language'] == 'English') | (battles['language'] == 'unknown')]
eng_battles, eng_battles_no_ties = subselect_battles(eng_battles, selected_models)

Now, let's view the top 10 promiment prompts for battles in English.
Note that there is a caveat for LMArena data, where some of the lanauges are labeled as 'unknown'. This is the reason why we have `battles['language'] == 'unknown'` as part our filter.

In [ ]:
# Print the top 10 most common prompts along with their count and percentage of total prompts
top_prompts = eng_battles["prompt"].value_counts().head(10)
for i, (prompt, count) in enumerate(top_prompts.items(), 1):
    print(f"Rank {i}: {count} samples ({round(100 * count/len(eng_battles), 2)}%)\n{prompt}\n")

# print the total percentage of prompts that are 1 of the top 10 prompts
top_10_percentage = sum(top_prompts) / len(eng_battles)
print(f"Total percentage of prompts that are 1 of the top 10 prompts: {round(100 * top_10_percentage, 2)}%")


## Question 3a

When evaluating models, it’s useful to understand which prompt types are over-represented and how these popular prompts can influence the leaderboard. Below we have plotted the leaderboard when all of the top 10 prompts are removed.

**Task:**  Compare the leaderboard generated from `get_pairwise_win_fraction_plot(eng_battles_no_ties_no_top_prompts)` to the leaderboard you generated in the previous problem. In 2-3 sentences, answer the following:

In [ ]:
# remove top prompts top_prompts from eng_battles_no_ties
eng_battles_no_ties_no_top_prompts = eng_battles_no_ties[~eng_battles_no_ties["prompt"].isin(top_prompts.index)]

pairwise_win_rate, fig = get_pairwise_win_fraction_plot(eng_battles_no_ties_no_top_prompts, title="Average Win Rate Against All Other Models (No Top Prompts)")
fig.show()


## Question 3b
LMArena also provides more detailed **category labels** inside the columns `is_code` and nested `category_tag` column.
We have already extracted the following boolean Series for you. Feel free to refer to these columns when you are analyzing your clusters for Question 3c.

**Task:**
1. Make a bar chart showing these proportion for each category (i.e. the fraction of battles where the category is True)
2. Using `get_pairwise_win_fraction_plot`, compute the pairwise win rate for each category and use the `plot_category_rank_heatmap` to visualize the results. Ensure that your data is in *tidy format*, containing columns `model`, `category`, and `win_rate`. You should have each of the categories above as well as an 'overall' category which is the `pairwise_win_rate` you computed previously.

In [ ]:
def plot_category_rank_heatmap(df: pd.DataFrame) -> None:
    """
    Plot a heatmap of model ranks by category.
    """
    assert "overall" in df["category"].unique(), "'overall' was not found as a category in your dataframe"
    rank_table = df.pivot(index="model", columns="category", values="rank")

    if "overall" in rank_table.columns:
        cols = ["overall"] + [c for c in rank_table.columns if c != "overall"]
        rank_table = rank_table[cols]
    # sort models by overall rank
    rank_table = rank_table.sort_values("overall", ascending=True)
    px.imshow(
        rank_table,
        text_auto=True,
        color_continuous_scale="Viridis_r",
        labels=dict(x="Category", y="Model", color="Rank (1=best)"),
        zmin=1, zmax=rank_table.max().max(),
        aspect="auto"
    ).update_layout(
        title="Overall and Per-Category Ranks",
        width=950,
        height=400 + 12 * len(rank_table),
        xaxis_side="top"
    ).show()

In [ ]:
# LMArena also provides more detailed category labels inside the columns `is_code`, `is_refusal`,
# and the nested `category_tag` column.
# We have already extracted the following boolean Series for you:

# GIVEN (do not modify)
expected_creative = eng_battles_no_ties['category_tag'].apply(lambda x: x['criteria_v0.1']['creativity'])
expected_tech = eng_battles_no_ties['category_tag'].apply(lambda x: x['criteria_v0.1']['technical_accuracy'])
expected_if = eng_battles_no_ties['category_tag'].apply(lambda x: x['if_v0.1']['if'])
expected_math = eng_battles_no_ties['category_tag'].apply(lambda x: x['math_v0.1']['math'])
expected_code = (eng_battles_no_ties['is_code'] == True)

# Task:
# 1) Make a bar chart showing these proportion for each category (i.e. the fraction of battles where the category is True)
# 2) Compute the pairwise win rate for each category and use the plot_category_rank_heatmap to visualize the results

# TODO: plot a bar chart of the proportions
    # TODO: Implement this section

# TODO: compute the pairwise win rate for each category and use the plot_category_rank_heatmap to visualize the results
# Ensure that your data is tidy format (i.e. has the columns model, category, and win_rate). We have added the category column to the pairwise_win_rate dataframe below.

pairwise_win_rate['category'] = 'overall'
for category in category_dataframes:
    # TODO: Implement this section

rank_dataframes = ...
plot_category_rank_heatmap(rank_dataframes)



You might notice that the leaderboards can change quite a bit! This is because different model developers often put more emphasis on ceratin tasks in training their LLMs to better cater to their audience. Many of these models are also limited by the amount of data and compute available, which can further force specialization as some tasks are much harder to learn (esoecially if the model is on the smaller side).

We will explore these ideas more in part 2 of the homework.

## Question 3c

The category labels provide a clean way to divide problems but there are likely other ways to group prompts which can reveal other common use cases.

We'll now dig deeper by discovering prompt topics via K-Means clustering on the **prompt text**. Since we are interested in seeing what kinda of questions people are asking, use the **no-top-prompts** subset from earlier (`eng_battles_no_ties_no_top_prompts`) to reduce noise. We have sampled 8,000 battles for reasonable runtime and provided a helper function for clustering.


In [ ]:
# General KMeans for any embedding
def kmeans_cluster_prompts(features: np.ndarray, prompts: np.ndarray, k: int, random_state: int = 42):
    """
    Perform k-means clustering on features and return:
      - dataframe with prompts and their cluster assignments
      - model inertia (float)
      - elapsed runtime (seconds)
    """
    t0 = time.perf_counter()
    km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
    cluster_labels = km.fit_predict(features)
    elapsed = time.perf_counter() - t0
    df = pd.DataFrame({"prompt": prompts, "cluster": cluster_labels})
    return df, km.inertia_, elapsed

# Note: The actual clustering implementation is in the next cell

In [ ]:
# Since we're focusing on the prompt text and removing top-prompt bias,
# use the no top prompts subset `eng_battles_sample`:

# 1) We already provided you eng_battles_sample with the 8000 samples
# 2) Build TF-IDF features X from eng_battles_sample["prompt"] (default max_features=500)
# 3) You can use kmeans_cluster_prompts(features, prompts, k, random_state) to return:
#       - DataFrame ['prompt','cluster'], inertia (float), elapsed seconds (float)
# 4) Sweep K over [4, 6, 8, 10, 12] collecting times and inertias
# 5) Plot runtime vs K and elbow (inertia vs K)
# 6) Choose a best_K (e.g., 8) from elbow
# 7) Assign labels to eng_battles_sample["cluster"]
# 8) Visualize with 2D projection (ex: using SVD) colored by cluster
# 9) Save the clustered df to clustered_prompts_no_top_prompts.csv

    # TODO: Implement this section
# Save to CSV
out_path = f"clustered_prompts_no_top_prompts_k{best_K}.csv"
YOUR_DF.to_csv(out_path, index=False)
print(f"Saved -> {out_path}")




## Question 3d

We’ll now move from training to interpretation. Using the cluster labels from your selected K, briefly characterize what at least one distinct cluster seems to contain, or explain why patterns are unclear and how you might improve them.

You can examine the saved clustered_prompts_no_top_prompts.csv file to inspect the prompts in each cluster.

**Task: Answer the questions below**